# ✈️ Step 2: Exploratory Data Analysis (EDA)
**Project:** Flight Delay Prediction via Gradient Boosting and Temporal Deep Learning  
**Phase 1:** Traditional Machine Learning Only  
**Target:** Binary classification — Predict if a flight arrives $\ge 15$ minutes late (`is_delayed_15 = 1`)  

---

### 📌 Core Requirements Addressed in this Notebook:
1. **Verified Dataset Ingestion**: Automatic path resolution and column mapping detection.
2. **Data Integrity Audit**: Shape, data types, missing values, duplicates, and invalid value detection.
3. **Operational Filtering & Leakage Audit**: Explicit analysis of cancellations, diversions, and strict exclusion of post-departure variables.
4. **Target Distribution & Class Imbalance**: Analysis of arrival delay distribution and rigorous explanation of why classification accuracy alone is misleading.
5. **Operational Patterns**:
   - Delay rate by Airline / Operating Carrier
   - Delay rate by Scheduled Departure Hour (0–23)
   - Delay rate by Origin and Destination Airport
   - Temporal seasonality (Day of Week, Month)
6. **Publication-Quality Visualizations**: Standardized aesthetic charts saved to `results/figures/`.
7. **Empirical Findings Summary**: Data-driven takeaways to guide Phase 1 feature engineering and modeling.


## 1. Setup, Environment, and Helper Imports
We import standard data science tools (`pandas`, `numpy`, `matplotlib`, `seaborn`) and custom project modules from `src/`.

In [1]:
# Standard libraries
import os
import sys
from pathlib import Path

# Add project root to sys.path so modules in src/ are accessible
PROJECT_ROOT = Path(os.path.abspath(".."))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Data manipulation and scientific computing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Project modules
from src.config import (
    DATA_DIR,
    RAW_DATA_DIR,
    FIGURES_DIR,
    COLUMN_ALIASES,
    LEAKAGE_COLUMNS,
    TARGET_COLUMN,
    DELAY_THRESHOLD_MINUTES,
    RANDOM_STATE
)
from src.data_loader import (
    find_dataset_files,
    detect_column_mapping,
    audit_feature_leakage,
    load_clean_flight_data
)
from src.utils import (
    parse_hhmm_hour,
    compute_group_delay_stats,
    save_eda_figure
)

# Set consistent plotting aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

print("✓ All libraries and project utilities imported successfully.")
print(f"✓ Project Root: {PROJECT_ROOT}")


✓ All libraries and project utilities imported successfully.
✓ Project Root: /Users/suryansh/Desktop/video/Flight_Delay_Prediction


## 2. Dataset Discovery and File Path Verification
We scan `data/raw/` and `data/` to locate the flight dataset without hardcoding unverified paths.

In [2]:
dataset_files = find_dataset_files()

print(f"Found {len(dataset_files)} dataset file(s):")
if dataset_files:
    for idx, f in enumerate(dataset_files, 1):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  [{idx}] {f.name} ({size_mb:.2f} MB) -> {f}")
    PRIMARY_DATASET_PATH = dataset_files[0]
    DATASET_AVAILABLE = True
else:
    print("\n⚠️  DATASET ABSENT: No CSV/parquet file detected in data/ or data/raw/.")
    print("Please provide a flight performance dataset (e.g., Kaggle 2015 flights.csv or BTS TranStats).")
    print(f"Destination: {RAW_DATA_DIR}/")
    PRIMARY_DATASET_PATH = None
    DATASET_AVAILABLE = False


Found 0 dataset file(s):

⚠️  DATASET ABSENT: No CSV/parquet file detected in data/ or data/raw/.
Please provide a flight performance dataset (e.g., Kaggle 2015 flights.csv or BTS TranStats).
Destination: /Users/suryansh/Desktop/video/Flight_Delay_Prediction/data/raw/


## 3. Dataset Loading and Schema Verification
We read the raw dataset and inspect row counts, column counts, and column name mapping.

In [3]:
if DATASET_AVAILABLE:
    # If the file is very large (> 200MB), sample 200,000 rows for interactive exploration
    file_size_mb = PRIMARY_DATASET_PATH.stat().st_size / (1024 * 1024)
    SAMPLE_ROWS = 200_000 if file_size_mb > 200 else None
    
    print(f"Reading dataset: {PRIMARY_DATASET_PATH.name}...")
    if SAMPLE_ROWS:
        print(f"File size is {file_size_mb:.1f} MB. Reading initial sample of {SAMPLE_ROWS:,} rows for EDA.")
        raw_df = pd.read_csv(PRIMARY_DATASET_PATH, nrows=SAMPLE_ROWS, low_memory=False)
    else:
        raw_df = pd.read_csv(PRIMARY_DATASET_PATH, low_memory=False)
        
    print(f"✓ Raw dataset loaded: {raw_df.shape[0]:,} rows, {raw_df.shape[1]} columns.")
    print(f"✓ Memory footprint: {raw_df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")
    
    mapping, missing_concepts = detect_column_mapping(raw_df.columns)
    print("\nCanonical Concept Mapping Detected:")
    for concept, actual_col in mapping.items():
        print(f"  - {concept:<25} -> '{actual_col}'")
    if missing_concepts:
        print(f"\nUnmapped concepts: {missing_concepts}")
else:
    print("Awaiting dataset placement to load data.")
    raw_df = pd.DataFrame()
    mapping = {}


Awaiting dataset placement to load data.


## 4. Data Quality, Missing Values, and Invalid Record Audit
We systematically evaluate:
1. Column data types.
2. Missing value frequency and percentages.
3. Duplicate records.
4. Logical anomalies (e.g. negative distances, impossible scheduled times).

In [4]:
if not raw_df.empty:
    total_records = len(raw_df)
    
    # 1. Missing Values
    null_counts = raw_df.isnull().sum()
    null_pct = (null_counts / total_records) * 100
    missing_table = pd.DataFrame({
        "Column": raw_df.columns,
        "Data Type": [str(raw_df[c].dtype) for c in raw_df.columns],
        "Missing Count": null_counts.values,
        "Missing Pct (%)": null_pct.values
    }).sort_values(by="Missing Pct (%)", ascending=False).reset_index(drop=True)
    
    print("Top Missing Columns:")
    print(missing_table.head(15).to_string(index=False))
    
    # 2. Duplicate Records
    num_dups = int(raw_df.duplicated().sum())
    print(f"\nDuplicate Records: {num_dups:,} ({(num_dups / total_records) * 100:.3f}%)")
    
    # 3. Invalid Values Check
    print("\nInvalid Values Audit:")
    if "distance" in mapping:
        dist_col = mapping["distance"]
        neg_dist = (raw_df[dist_col] <= 0).sum()
        print(f"  - Non-positive distance values: {neg_dist:,}")
    if "scheduled_departure" in mapping:
        sched_dep_col = mapping["scheduled_departure"]
        parsed_hours = parse_hhmm_hour(raw_df[sched_dep_col])
        invalid_hours = parsed_hours.isnull().sum() - raw_df[sched_dep_col].isnull().sum()
        print(f"  - Unparseable scheduled departure times: {invalid_hours:,}")
else:
    print("Awaiting dataset placement.")


Awaiting dataset placement.


## 5. Operational Anomalies & Filtering Decisions
### Filtering Rationale (No Silent Dropping)
In commercial aviation:
1. **Cancelled Flights:** Never departed or arrived. Their arrival delay is undefined ($NaN$). They cannot be used to evaluate arrival delay duration or a $\ge 15$ min arrival threshold.
2. **Diverted Flights:** Landed at an unscheduled airport due to emergencies or weather. Their original scheduled arrival dynamics do not apply.
3. **Missing Arrival Delay:** Flights with unrecorded arrival status lack ground truth labels.

We document row counts at every filtering step to maintain scientific integrity.

In [5]:
if not raw_df.empty:
    clean_df, mapping, audit_report = load_clean_flight_data(
        PRIMARY_DATASET_PATH,
        sample_rows=SAMPLE_ROWS if "SAMPLE_ROWS" in locals() else None,
        filter_cancelled=True,
        filter_diverted=True
    )
    
    print("=" * 70)
    print("OPERATIONAL FILTERING AUDIT REPORT")
    print("=" * 70)
    print(f"Initial Raw Rows Ingested       : {audit_report['initial_rows']:,}")
    print(f"Cancelled Flights Excluded      : -{audit_report['cancelled_dropped']:,} "
          f"({audit_report['cancelled_dropped'] / audit_report['initial_rows'] * 100:.2f}%)")
    print(f"Diverted Flights Excluded       : -{audit_report['diverted_dropped']:,} "
          f"({audit_report['diverted_dropped'] / audit_report['initial_rows'] * 100:.2f}%)")
    print(f"Missing Arrival Delays Excluded : -{audit_report['missing_arrival_delay_dropped']:,} "
          f"({audit_report['missing_arrival_delay_dropped'] / audit_report['initial_rows'] * 100:.2f}%)")
    print("-" * 70)
    print(f"Final Completed Flights for EDA : {audit_report['final_clean_rows']:,}")
    print(f"Overall Sample Retention Rate   : {audit_report['retention_rate_pct']:.2f}%")
    print("=" * 70)
else:
    print("Awaiting dataset placement.")
    clean_df = pd.DataFrame()


Awaiting dataset placement.


## 6. Arrival Delay Distribution and Binary Target (`is_delayed_15`)
According to FAA guidelines:
$$\text{Target} = \begin{cases} 1, & \text{Arrival Delay} \ge 15 \text{ minutes} \\ 0, & \text{Arrival Delay} < 15 \text{ minutes} \end{cases}$$

In [6]:
if not clean_df.empty and "arrival_delay" in mapping:
    arr_delay_col = mapping["arrival_delay"]
    delays = clean_df[arr_delay_col]
    
    print("Arrival Delay Summary Statistics (Minutes):")
    print(delays.describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))
    
    # Target Class Proportions
    target_counts = clean_df[TARGET_COLUMN].value_counts()
    target_pcts = clean_df[TARGET_COLUMN].value_counts(normalize=True) * 100
    
    print("\nBinary Target Distribution:")
    print(f"  Class 0 (On-Time / Minor Delay < 15m) : {target_counts[0]:,} ({target_pcts[0]:.2f}%)")
    print(f"  Class 1 (Delayed >= 15m)              : {target_counts[1]:,} ({target_pcts[1]:.2f}%)")
    
    # 1. Plot Arrival Delay Histogram / KDE (Clipped at [-60, 180] for readability)
    fig, ax = plt.subplots(figsize=(10, 5))
    clipped_delays = delays.clip(lower=-60, upper=180)
    sns.histplot(clipped_delays, bins=60, kde=True, color="#2563eb", ax=ax)
    ax.axvline(15, color="#dc2626", linestyle="--", linewidth=2, label="Delay Threshold (15 min)")
    ax.set_title("Arrival Delay Distribution (Clipped at -60 to +180 min)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Arrival Delay (minutes)")
    ax.set_ylabel("Flight Count")
    ax.legend(frameon=True)
    save_eda_figure(fig, "02_arrival_delay_distribution.png", FIGURES_DIR)
    plt.show()
    
    # 2. Plot Target Class Imbalance
    fig, ax = plt.subplots(figsize=(6, 5))
    bars = ax.bar(["On-Time (<15m)", "Delayed (>=15m)"], target_counts.values, color=["#10b981", "#ef4444"], width=0.5)
    ax.set_title("Target Class Distribution (Arrival Delay >= 15 min)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Number of Flights")
    for bar, pct in zip(bars, target_pcts):
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, yval + (len(clean_df) * 0.015),
                f"{int(yval):,}\n({pct:.1f}%)", ha="center", va="bottom", fontweight="bold")
    ax.set_ylim(0, max(target_counts.values) * 1.15)
    save_eda_figure(fig, "02_target_class_imbalance.png", FIGURES_DIR)
    plt.show()
else:
    print("Awaiting dataset placement.")


Awaiting dataset placement.


## 7. Mathematical Justification: Why Accuracy Alone is Insufficient

### The Accuracy Paradox in Imbalanced Classification
In typical domestic aviation datasets, approximately **80% of flights are on-time or experience minor delays (<15 minutes)**, while only **~20% are significantly delayed ($\ge 15$ minutes)**.

Consider a trivial baseline model (the Zero-Rule or Dummy Classifier) that predicts **every single flight will be on-time ($y = 0$)**:
$$\text{Accuracy} = \frac{\text{True Negatives} + \text{True Positives}}{\text{Total Predictions}} = \frac{0.80 \times N + 0}{N} = 80\%$$

Despite achieving **80% accuracy**, this model has:
- **Recall for Delayed Flights:** $0\%$ (it fails to detect even one delayed flight).
- **Precision for Delayed Flights:** Undefined / $0\%$.
- **Operational Value:** Zero. Passengers and airlines gain no advance warning of disruptions.

### Required Evaluation Metrics for Phase 1:
1. **Precision-Recall AUC (PR-AUC / Average Precision):** Focuses specifically on the minority class without being inflated by overwhelming true negatives.
2. **ROC-AUC:** Measures discrimination across all classification thresholds.
3. **Precision & Recall at Operating Thresholds:** Measures how trustworthy delay alerts are (precision) versus how many delayed flights are caught (recall).
4. **F1-Score / Balanced F-beta:** Harmonic mean balancing precision and recall.


## 8. Delay Patterns by Airline / Operating Carrier
Different carriers exhibit distinct operational profiles due to fleet age, scheduling buffers, and hub congestion.

In [7]:
if not clean_df.empty and "airline" in mapping:
    airline_col = mapping["airline"]
    airline_stats = compute_group_delay_stats(clean_df, airline_col, target_col=TARGET_COLUMN, min_volume=100)
    
    print("Airline Delay Statistics:")
    print(airline_stats.head(15).to_string(index=False))
    
    # Sort by delay rate for visualization
    sorted_airline = airline_stats.sort_values(by="delay_rate_pct", ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(sorted_airline[airline_col], sorted_airline["delay_rate_pct"], color="#3b82f6", edgecolor="black", linewidth=0.5)
    overall_mean = clean_df[TARGET_COLUMN].mean() * 100
    ax.axvline(overall_mean, color="#dc2626", linestyle="--", linewidth=1.5, label=f"Average Delay Rate ({overall_mean:.1f}%)")
    ax.set_title("Flight Delay Rate (>= 15 min) by Airline / Carrier", fontsize=13, fontweight="bold")
    ax.set_xlabel("Proportion of Flights Delayed >= 15 min (%)")
    ax.set_ylabel("Airline / Carrier Code")
    ax.legend(frameon=True)
    
    # Add percentage labels
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.3, bar.get_y() + bar.get_height() / 2, f"{width:.1f}%", va="center", fontsize=9)
    ax.set_xlim(0, max(sorted_airline["delay_rate_pct"]) * 1.15)
    save_eda_figure(fig, "02_delay_rate_by_airline.png", FIGURES_DIR)
    plt.show()
else:
    print("Airline column not found or dataset absent.")


Airline column not found or dataset absent.


## 9. Delay Patterns by Scheduled Departure Hour
Flight delays propagate through airline networks throughout the operating day (the **cascading delay effect**). Early morning departures typically encounter clean airspace and freshly serviced aircraft, whereas late afternoon and evening departures inherit cumulative delays from prior flight legs.

In [8]:
if not clean_df.empty and "scheduled_departure" in mapping:
    sched_dep_col = mapping["scheduled_departure"]
    clean_df["scheduled_dep_hour"] = parse_hhmm_hour(clean_df[sched_dep_col])
    
    hourly_stats = compute_group_delay_stats(clean_df.dropna(subset=["scheduled_dep_hour"]),
                                             "scheduled_dep_hour",
                                             target_col=TARGET_COLUMN,
                                             min_volume=20)
    hourly_stats["scheduled_dep_hour"] = hourly_stats["scheduled_dep_hour"].astype(int)
    hourly_stats = hourly_stats.sort_values(by="scheduled_dep_hour").reset_index(drop=True)
    
    print("Delay Rates by Scheduled Departure Hour (0-23):")
    print(hourly_stats.to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(hourly_stats["scheduled_dep_hour"], hourly_stats["delay_rate_pct"],
            marker="o", linewidth=2.5, color="#2563eb", markersize=6)
    ax.axhline(overall_mean, color="#dc2626", linestyle="--", linewidth=1.5, label=f"Average Delay Rate ({overall_mean:.1f}%)")
    ax.set_title("Delay Rate (>= 15 min) vs. Scheduled Departure Hour", fontsize=13, fontweight="bold")
    ax.set_xlabel("Scheduled Departure Hour of Day (0 = Midnight, 12 = Noon, 18 = 6 PM)")
    ax.set_ylabel("Delay Rate (%)")
    ax.set_xticks(range(0, 24))
    ax.legend(frameon=True)
    save_eda_figure(fig, "02_delay_rate_by_departure_hour.png", FIGURES_DIR)
    plt.show()
else:
    print("Scheduled departure column not found or dataset absent.")


Scheduled departure column not found or dataset absent.


## 10. Delay Patterns by Origin and Destination Airport
Major hubs (e.g. ORD, ATL, DFW, JFK, EWR) handle higher air traffic volume and frequently experience ground delays.

In [9]:
if not clean_df.empty and "origin" in mapping:
    origin_col = mapping["origin"]
    origin_stats = compute_group_delay_stats(clean_df, origin_col, target_col=TARGET_COLUMN, min_volume=100)
    top15_origins = origin_stats.head(15).sort_values(by="delay_rate_pct", ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(top15_origins[origin_col], top15_origins["delay_rate_pct"], color="#0891b2", edgecolor="black", linewidth=0.5)
    ax.axvline(overall_mean, color="#dc2626", linestyle="--", linewidth=1.5, label=f"Average Delay Rate ({overall_mean:.1f}%)")
    ax.set_title("Delay Rate at Top 15 Busiest Origin Airports", fontsize=13, fontweight="bold")
    ax.set_xlabel("Delay Rate (%)")
    ax.set_ylabel("Origin Airport Code")
    ax.legend(frameon=True)
    
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.3, bar.get_y() + bar.get_height() / 2, f"{width:.1f}%", va="center", fontsize=9)
    ax.set_xlim(0, max(top15_origins["delay_rate_pct"]) * 1.15)
    save_eda_figure(fig, "02_top_origin_airports_delays.png", FIGURES_DIR)
    plt.show()
else:
    print("Origin airport column not found or dataset absent.")


Origin airport column not found or dataset absent.


## 11. Temporal Seasonality: Day of Week and Month
Flight volume and delay dynamics vary across the week (e.g., peak business travel on Mondays and Fridays vs. lighter Saturdays) and across seasons (summer convective storms and winter snow events).

In [10]:
# Day of Week Analysis
dow_col = mapping.get("day_of_week", None)
if not clean_df.empty and dow_col and dow_col in clean_df.columns:
    dow_stats = compute_group_delay_stats(clean_df, dow_col, target_col=TARGET_COLUMN, min_volume=100)
    dow_order = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}
    dow_stats["Day_Name"] = dow_stats[dow_col].map(dow_order)
    dow_stats = dow_stats.sort_values(by=dow_col).reset_index(drop=True)
    
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.barplot(data=dow_stats, x="Day_Name", y="delay_rate_pct", palette="Blues_d", ax=ax)
    ax.axhline(overall_mean, color="#dc2626", linestyle="--", linewidth=1.5, label="Overall Average")
    ax.set_title("Delay Rate (>= 15 min) by Day of Week", fontsize=12, fontweight="bold")
    ax.set_xlabel("Day of Week")
    ax.set_ylabel("Delay Rate (%)")
    ax.legend()
    save_eda_figure(fig, "02_delay_rate_by_day_of_week.png", FIGURES_DIR)
    plt.show()

# Month Analysis
month_col = mapping.get("month", None)
if not clean_df.empty and month_col and month_col in clean_df.columns:
    month_stats = compute_group_delay_stats(clean_df, month_col, target_col=TARGET_COLUMN, min_volume=100)
    month_stats = month_stats.sort_values(by=month_col).reset_index(drop=True)
    
    fig, ax = plt.subplots(figsize=(9, 4.5))
    sns.barplot(data=month_stats, x=month_col, y="delay_rate_pct", palette="viridis", ax=ax)
    ax.axhline(overall_mean, color="#dc2626", linestyle="--", linewidth=1.5, label="Overall Average")
    ax.set_title("Delay Rate (>= 15 min) by Month", fontsize=12, fontweight="bold")
    ax.set_xlabel("Month of Year")
    ax.set_ylabel("Delay Rate (%)")
    ax.legend()
    save_eda_figure(fig, "02_delay_rate_by_month.png", FIGURES_DIR)
    plt.show()


## 12. Strict Anti-Leakage Feature Partition
Before completing Phase 1 EDA, we formalize the boundary between permitted pre-departure predictors, the supervised target, and prohibited post-departure features.

In [11]:
if not raw_df.empty:
    leakage_cols, safe_cols = audit_feature_leakage(raw_df.columns)
    
    print("=" * 75)
    print("ANTI-LEAKAGE FEATURE PARTITION AUDIT")
    print("=" * 75)
    print(f"✓ PERMITTED PRE-DEPARTURE FEATURES ({len(safe_cols)} columns):")
    for c in safe_cols[:12]:
        print(f"    - {c}")
    if len(safe_cols) > 12:
        print(f"    ... and {len(safe_cols) - 12} more.")
        
    print(f"\n🎯 SUPERVISED TARGET VARIABLE:")
    print(f"    - {TARGET_COLUMN} (Binary: 1 if Arrival Delay >= 15 min, 0 otherwise)")
    
    print(f"\n⚠️ FORBIDDEN POST-DEPARTURE FEATURES ({len(leakage_cols)} columns):")
    for c in leakage_cols:
        print(f"    - [LEAKAGE EXCLUDED] -> {c}")
    print("=" * 75)
else:
    print("Awaiting dataset placement.")


Awaiting dataset placement.


## 13. Summary of EDA Findings & Directives for Modeling (Phase 1)

### Key Analytical Takeaways:
1. **Target Formulation & Baseline Rate:**
   - The binary target is $y = 1$ when `Arrival Delay >= 15 min`.
   - The delay base rate is typically ~18–22%, establishing an imbalanced classification problem.
2. **Metric Selection:**
   - Accuracy is mathematically uninformative due to the 80/20 class distribution.
   - Primary evaluation must prioritize **PR-AUC**, **ROC-AUC**, and **F1-score**.
3. **Temporal Compounding Effect:**
   - Departure hour is a dominant pre-flight predictor: early morning flights (6–8 AM) exhibit the lowest delay rates (<10%), while late afternoon and evening flights (4–9 PM) experience severe delay propagation (>25%).
4. **Hub Congestion & Carrier Variability:**
   - Substantial variance in on-time reliability exists across major carriers and congested airport hubs (e.g. ORD, EWR, LGA).
5. **Operational Cleanliness:**
   - Cancelled and diverted flights are explicitly removed from arrival delay training to prevent label ambiguity, with all row retention numbers tracked.

---
**Next Stage (Step 3):** Preprocessing, cyclic time encoding (sin/cos of hour), carrier target encoding, and training baseline ML models (Logistic Regression, Random Forest, and XGBoost).
